In [1]:
import pandas as pd
import geopandas as gpd
from shapely.wkt import loads
from shapely.ops import unary_union
import numpy as np
import json
from shapely import wkt

### Hotels

In [118]:
hotels_tt = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/hotels_traveltime.csv")
space_syntax = gpd.read_file("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/new_space_syntax_pst.gpkg")

In [119]:
#25min 2000m
hotels_tt["walking_25min"] = hotels_tt["walking_25min"].apply(lambda x: loads(x))
hotels_tt_25min = gpd.GeoDataFrame(hotels_tt, geometry = "walking_25min", crs = "EPSG:4326")[["place_id", "name", "walking_25min"]]
hotels_tt_25min = hotels_tt_25min.to_crs("EPSG:28992")
#15min 1200m
hotels_tt["walking_15min"] = hotels_tt["walking_15min"].apply(lambda x: loads(x))
hotels_tt_15min = gpd.GeoDataFrame(hotels_tt, geometry = "walking_15min", crs = "EPSG:4326")[["place_id", "name", "walking_15min"]]
hotels_tt_15min = hotels_tt_15min.to_crs("EPSG:28992")
#5min 400m
hotels_tt["walking_5min"] = hotels_tt["walking_5min"].apply(lambda x: loads(x))
hotels_tt_5min = gpd.GeoDataFrame(hotels_tt, geometry = "walking_5min", crs = "EPSG:4326")[["place_id", "name", "walking_5min"]]
hotels_tt_5min = hotels_tt_5min.to_crs("EPSG:28992")

In [23]:
hotels_tt_5min

,place_id,name,walking_5min
0,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,"MULTIPOLYGON (((120469.393 486772.663, 120469...."
1,ChIJIYdU8sMJxkcRrbxZuGZOlro,The Hoxton Amsterdam,"MULTIPOLYGON (((120622.466 487263.316, 120628...."
2,ChIJozDTD8MJxkcRO_HxQ1B4wF4,Grand Canal Boutique Hotel,"MULTIPOLYGON (((120447.398 486972.134, 120437...."
3,ChIJz-L17MMJxkcRZdrqv-u57P4,Hotel HEGRA by Stanley Collection,"MULTIPOLYGON (((120672.946 487244.162, 120682...."
4,ChIJo8-ANMIJxkcRi1HeWcbBcmI,Ambassade Hotel - Amsterdam,"MULTIPOLYGON (((120722.674 486970.6, 120722.60..."
...,...,...,...
414,ChIJPcEtxHEKxkcRx84qVvhT_uE,Cityden Zuidas,"MULTIPOLYGON (((119771.99 481298.257, 119771.8..."
415,ChIJVWZIVwsKxkcR4vauw0vYxuk,"Holiday Inn Express Amsterdam - South, an IHG ...","MULTIPOLYGON (((119270.415 481998.926, 119310...."
416,ChIJC_KvV9jhxUcRgDzDAbDHkBw,Amsterdam Forest Hotel,"MULTIPOLYGON (((118720.524 480896.844, 118720...."
417,ChIJN8RMMqgLxkcRknxrSHEle1E,Aparthotel Adagio Amsterdam City South,"MULTIPOLYGON (((119710.081 481343.172, 119740...."


In [4]:
space_syntax_h2000 = space_syntax[space_syntax['AC_w2k_NACH'] >= space_syntax["AC_w2k_NACH"].quantile(0.75)]
space_syntax_h1200 = space_syntax[space_syntax['AC_w1200_NACH'] >= space_syntax["AC_w1200_NACH"].quantile(0.75)]
space_syntax_h400 = space_syntax[space_syntax['AC_w400_NACH'] >= space_syntax["AC_w400_NACH"].quantile(0.75)]

In [5]:
# Getting the lines in every polygon and all high scoring lines on A.C.
hotels_clip_5min = hotels_tt_5min.overlay(space_syntax, how = "intersection", keep_geom_type = False)
hotels_clip_5min_h = hotels_tt_5min.overlay(space_syntax_h400, how = "intersection", keep_geom_type = False)

hotels_clip_15min = hotels_tt_15min.overlay(space_syntax, how = "intersection", keep_geom_type = False)
hotels_clip_15min_h = hotels_tt_15min.overlay(space_syntax_h1200, how = "intersection", keep_geom_type = False)

hotels_clip_25min = hotels_tt_25min.overlay(space_syntax, how = "intersection", keep_geom_type = False)
hotels_clip_25min_h = hotels_tt_25min.overlay(space_syntax_h2000, how = "intersection", keep_geom_type = False)

In [6]:
#calculting length of network and length of high A.C. network
hotels_clip_5min_h["length line high"] = hotels_clip_5min_h["geometry"].length
hotels_clip_5min["length line"] = hotels_clip_5min["geometry"].length
hotels_5min_h_length = hotels_clip_5min_h.groupby(["place_id", "name"])["length line high"].sum().reset_index()
hotels_5min_length = hotels_clip_5min.groupby(["place_id", "name"])["length line"].sum().reset_index()

hotels_clip_15min_h["length line high"] = hotels_clip_15min_h["geometry"].length
hotels_clip_15min["length line"] = hotels_clip_15min["geometry"].length
hotels_15min_h_length = hotels_clip_15min_h.groupby(["place_id", "name"])["length line high"].sum().reset_index()
hotels_15min_length = hotels_clip_15min.groupby(["place_id", "name"])["length line"].sum().reset_index()

hotels_clip_25min_h["length line high"] = hotels_clip_25min_h["geometry"].length
hotels_clip_25min["length line"] = hotels_clip_25min["geometry"].length
hotels_25min_h_length = hotels_clip_25min_h.groupby(["place_id", "name"])["length line high"].sum().reset_index()
hotels_25min_length = hotels_clip_25min.groupby(["place_id", "name"])["length line"].sum().reset_index()

In [7]:
hotels_5min_length = hotels_5min_h_length.merge(hotels_5min_length, on = ["place_id", "name"], how = "inner")
hotels_15min_length = hotels_15min_h_length.merge(hotels_15min_length, on = ["place_id", "name"], how = "inner")
hotels_25min_length = hotels_25min_h_length.merge(hotels_25min_length, on = ["place_id", "name"], how = "inner")

In [8]:
hotels_5min_length["5min % high A.C."] = hotels_5min_length["length line high"] / hotels_5min_length["length line"] * 100
hotels_15min_length["15min % high A.C."] = hotels_15min_length["length line high"] / hotels_15min_length["length line"] * 100
hotels_25min_length["25min % high A.C."] = hotels_25min_length["length line high"] / hotels_25min_length["length line"] * 100

In [9]:
hotels_5min_length = hotels_5min_length.drop(columns = ["length line high", "length line"])
hotels_15min_length = hotels_15min_length.drop(columns = ["length line high", "length line"])
hotels_25min_length = hotels_25min_length.drop(columns = ["length line high", "length line"])

In [120]:
hotels_AC = hotels_5min_length.merge(hotels_15min_length, on = ["place_id", "name"], how = "inner")
hotels_AC = hotels_AC.merge(hotels_25min_length, on = ["place_id", "name"], how = "inner")
hotels_AC

,place_id,name,5min % high A.C.,15min % high A.C.,25min % high A.C.
0,ChIJ-1hUL74JxkcRnLpTBTbfH5o,Eden Hotel Amsterdam,18.456850,23.195140,26.018651
1,ChIJ-8C0ZsQJxkcR6f0pc6CA3IM,Hotel De Westertoren,8.903487,23.484635,25.645533
2,ChIJ-8DbTpYJxkcRF6GXegHzFe8,Hotel Hermitage Amsterdam,15.403297,23.963324,26.182952
3,ChIJ-RAyRD0KxkcRG4koK51opDE,Hotel Novotel Amsterdam City,24.843087,27.637239,27.644549
4,ChIJ-RS-x8cJxkcRix6HMEVbC0c,Avenue Hotel,8.965566,22.674889,25.467650
...,...,...,...,...,...
412,ChIJzUkvQZUJxkcR7vLn27sChg0,SWEETS hotel Westerdoksbrug,13.396704,21.903330,23.892418
413,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,30.882961,30.704666,27.029511
414,ChIJzVtZ7PkJxkcRFn8elAS0PHg,The Delphi - Amsterdam Townhouse,21.154826,21.546693,29.000832
415,ChIJzcu9uuUJxkcRsinjE9Hf-L4,Sonder Park House,13.452959,27.180408,28.292420


In [105]:
# Read data with info about locations and time
times = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/times.csv")
coordinates_places = times[["place_id", "coordinates"]]

# Parse the 'coordinates' column into separate latitude and longitude columns
times[['latitude', 'longitude']] = coordinates_places['coordinates'].apply(
    lambda x: pd.Series(json.loads(x))
)

# Drop the original 'coordinates' column for clarity
times = times.drop(columns='coordinates')

# Load the file with neighbourhood info
buurt = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/INDELING_BUURT (1).csv", delimiter=';')[["Buurt", "Wijk", "WKT_LNG_LAT"]]


# Load the points CSV
times_gdf = gpd.GeoDataFrame(
    times,
    geometry=gpd.points_from_xy(times['longitude'], times['latitude']),
    crs="EPSG:4326"  # Ensure the correct CRS, usually WGS84
)

# Load the polygons CSV
buurt['geometry'] = buurt['WKT_LNG_LAT'].apply(wkt.loads)
buurt_gdf = gpd.GeoDataFrame(buurt, geometry='geometry', crs="EPSG:4326")

# Perform spatial join
times_gdf = gpd.sjoin(times_gdf, buurt_gdf, how="left", predicate="within")
times_gdf = times_gdf.drop(columns = ["index_right", ])
times_gdf = times_gdf[["place_id", "name", "Buurt", "Wijk", "geometry", "WKT_LNG_LAT"]]

In [121]:
hotels_AC = hotels_AC.merge(times_gdf, on = ["place_id", "name"], how = "inner")
hotels_AC

,place_id,name,5min % high A.C.,15min % high A.C.,25min % high A.C.,Buurt,Wijk,geometry,WKT_LNG_LAT
0,ChIJ-1hUL74JxkcRnLpTBTbfH5o,Eden Hotel Amsterdam,18.456850,23.195140,26.018651,Rembrandtplein e.o.,Grachtengordel-Zuid,POINT (4.89869 52.36684),"POLYGON((4.892818 52.36518,4.895874 52.364754,..."
1,ChIJ-8C0ZsQJxkcR6f0pc6CA3IM,Hotel De Westertoren,8.903487,23.484635,25.645533,Felix Meritisbuurt,Grachtengordel-West,POINT (4.88614 52.37346),"POLYGON((4.882614 52.368932,4.884411 52.368862..."
2,ChIJ-8DbTpYJxkcRF6GXegHzFe8,Hotel Hermitage Amsterdam,15.403297,23.963324,26.182952,Weesperbuurt,Weesperbuurt/Plantage,POINT (4.90361 52.36444),"POLYGON((4.901217 52.365779,4.901557 52.365162..."
3,ChIJ-RAyRD0KxkcRG4koK51opDE,Hotel Novotel Amsterdam City,24.843087,27.637239,27.644549,De Klenckebuurt,Buitenveldert-Oost,POINT (4.88852 52.33372),"POLYGON((4.878652 52.335088,4.879167 52.334426..."
4,ChIJ-RS-x8cJxkcRix6HMEVbC0c,Avenue Hotel,8.965566,22.674889,25.467650,Nieuwendijk-Noord,Burgwallen-Nieuwe Zijde,POINT (4.89465 52.37665),"POLYGON((4.893582 52.376221,4.8951 52.376206,4..."
...,...,...,...,...,...,...,...,...,...
412,ChIJzUkvQZUJxkcR7vLn27sChg0,SWEETS hotel Westerdoksbrug,13.396704,21.903330,23.892418,Westerdokseiland,Haarlemmerbuurt,POINT (4.89335 52.38333),"POLYGON((4.890186 52.382395,4.890637 52.382179..."
413,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,30.882961,30.704666,27.029511,Riekerpolder,Sloten/Nieuw-Sloten,POINT (4.8248 52.34177),"POLYGON((4.819922 52.33852,4.832922 52.338426,..."
414,ChIJzVtZ7PkJxkcRFn8elAS0PHg,The Delphi - Amsterdam Townhouse,21.154826,21.546693,29.000832,Minervabuurt-Noord,Apollobuurt,POINT (4.87693 52.35043),"POLYGON((4.866777 52.350394,4.867401 52.348677..."
415,ChIJzcu9uuUJxkcRsinjE9Hf-L4,Sonder Park House,13.452959,27.180408,28.292420,P.C. Hooftbuurt,Museumkwartier,POINT (4.87785 52.36014),"POLYGON((4.877154 52.360101,4.878568 52.358153..."


In [158]:
#space syntax high values
#space_syntax_h400[["id", "AC_w400_NACH", "geometry"]].to_file("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/space_syntax_h400.gpkg", driver = "GPKG")
#space_syntax_h1200[["id", "AC_w1200_NACH", "geometry"]].to_file("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/space_syntax_h1200.gpkg", driver = "GPKG")
#space_syntax_h2000[["id", "AC_w2k_NACH", "geometry"]].to_file("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/space_syntax_h2000.gpkg", driver = "GPKG")

In [107]:
prs = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/pressure_tourism.csv", index_col=0)
plc_prs = prs.groupby(["place_id", "name"])["pressure"].sum().reset_index()
plc_prs = plc_prs.merge(times_gdf, on = ["place_id", "name"])
plc_prs = gpd.GeoDataFrame(plc_prs, geometry = "geometry", crs = "EPSG:4326")
plc_prs = plc_prs.to_crs("EPSG:28992")

In [122]:
#hotels_clip_5min = hotels_tt_5min.overlay(plc_prs, how = "intersection", keep_geom_type = False)
hotels_pressure_5min = gpd.sjoin(plc_prs, hotels_tt_5min, how="inner", predicate="within", lsuffix = "place", rsuffix = "hotel")
hotels_pressure_15min = gpd.sjoin(plc_prs, hotels_tt_15min, how="inner", predicate="within", lsuffix = "place", rsuffix = "hotel")
hotels_pressure_25min = gpd.sjoin(plc_prs, hotels_tt_25min, how="inner", predicate="within", lsuffix = "place", rsuffix = "hotel")


In [133]:
# Add hotel_id if not present
hotels_tt_5min["index_hotel"] = hotels_tt_5min.index
# Group by hotel polygon and sum the pressure
pressure_sums_5min = hotels_pressure_5min.groupby("index_hotel")["pressure"].sum().reset_index()
# Add hotels with no pressure
hotels_pressure_5min_h = hotels_tt_5min.merge(pressure_sums_5min, on="index_hotel", how="left")
hotels_pressure_5min_h["pressure 5min"] = hotels_pressure_5min_h["pressure"].fillna(0)


# Add hotel_id if not present
hotels_tt_15min["index_hotel"] = hotels_tt_15min.index
# Group by hotel polygon and sum the pressure
pressure_sums_15min = hotels_pressure_15min.groupby("index_hotel")["pressure"].sum().reset_index()
# Add hotels with no pressure
hotels_pressure_15min_h = hotels_tt_15min.merge(pressure_sums_15min, on="index_hotel", how="left")
hotels_pressure_15min_h["pressure 15min"] = hotels_pressure_15min_h["pressure"].fillna(0)

# Add hotel_id if not present
hotels_tt_25min["index_hotel"] = hotels_tt_25min.index
# Group by hotel polygon and sum the pressure
pressure_sums_25min = hotels_pressure_25min.groupby("index_hotel")["pressure"].sum().reset_index()
# Add hotels with no pressure
hotels_pressure_25min_h = hotels_tt_25min.merge(pressure_sums_25min, on="index_hotel", how="left")
hotels_pressure_25min_h["pressure 25min"] = hotels_pressure_25min_h["pressure"].fillna(0)



In [138]:
hotels_AC = hotels_AC.merge(hotels_pressure_5min_h, on = ["place_id", "name"], how = "inner") 
hotels_AC = hotels_AC.merge(hotels_pressure_15min_h, on = ["place_id", "name"], how = "inner") 
hotels_AC = hotels_AC.merge(hotels_pressure_25min_h, on = ["place_id", "name"], how = "inner")

In [140]:
data = hotels_AC
# Define a function to categorize clearly based on percentiles
def categorize(series):
    low = series.quantile(0.33)
    high = series.quantile(0.66)
    
    def get_category(value):
        if value <= low:
            return 'Low'
        elif value <= high:
            return 'Medium'
        else:
            return 'High'
    
    return series.apply(get_category)

# Categorize % high Angular Choice clearly
for col in ['5min % high A.C.', '15min % high A.C.', '25min % high A.C.']:
    data[f'{col} Category'] = categorize(data[col])

# Categorize Tourist Pressure clearly
for col in ['pressure 5min', 'pressure 15min', 'pressure 25min']:
    data[f'{col} Category'] = categorize(data[col])

# Define function to create combined categories clearly
def combined_category(ac, pressure):
    if ac == 'High' and pressure == 'Low':
        return 'High AC + Low Pressure (Potential)'
    elif ac == 'High' and pressure == 'High':
        return 'High AC + High Pressure (Stress)'
    elif ac == 'Low' and pressure == 'High':
        return 'Low AC + High Pressure (Critical)'
    elif ac == 'Medium' and pressure == 'Medium':
        return 'Medium AC + Medium Pressure (Balanced)'
    elif ac == 'Low' and pressure == 'Low':
        return 'Low AC + Low Pressure (Limited Interest)'
    elif ac == 'High' and pressure == 'Medium':
        return 'High AC + Medium Pressure (Manageable)'
    elif ac == 'Medium' and pressure == 'High':
        return 'Medium AC + High Pressure (Caution)'
    elif ac == 'Medium' and pressure == 'Low':
        return 'Medium AC + Low Pressure (Opportunity)'
    elif ac == 'Low' and pressure == 'Medium':
        return 'Low AC + Medium Pressure (Monitor)'
    else:
        return 'Other'

# Apply this combined category clearly for each timescale
for time in ['5min', '15min', '25min']:
    ac_col = f'{time} % high A.C. Category'
    pressure_col = f'pressure {time} Category'
    data[f'Combined Category {time}'] = data.apply(lambda row: combined_category(row[ac_col], row[pressure_col]), axis=1)
    

# Display the first rows to check categorization
data


,place_id,name,5min % high A.C.,15min % high A.C.,25min % high A.C.,Buurt,Wijk,geometry,WKT_LNG_LAT,walking_5min,...,pressure 25min,5min % high A.C. Category,15min % high A.C. Category,25min % high A.C. Category,pressure 5min Category,pressure 15min Category,pressure 25min Category,Combined Category 5min,Combined Category 15min,Combined Category 25min
0,ChIJ-1hUL74JxkcRnLpTBTbfH5o,Eden Hotel Amsterdam,18.456850,23.195140,26.018651,Rembrandtplein e.o.,Grachtengordel-Zuid,POINT (4.89869 52.36684),"POLYGON((4.892818 52.36518,4.895874 52.364754,...","MULTIPOLYGON (((121474.683 486597.511, 121484....",...,126486.580990,Medium,Medium,Medium,Medium,Medium,High,Medium AC + Medium Pressure (Balanced),Medium AC + Medium Pressure (Balanced),Medium AC + High Pressure (Caution)
1,ChIJ-8C0ZsQJxkcR6f0pc6CA3IM,Hotel De Westertoren,8.903487,23.484635,25.645533,Felix Meritisbuurt,Grachtengordel-West,POINT (4.88614 52.37346),"POLYGON((4.882614 52.368932,4.884411 52.368862...","MULTIPOLYGON (((120673.337 487410.513, 120678....",...,118663.332018,Low,Medium,Medium,Medium,High,High,Low AC + Medium Pressure (Monitor),Medium AC + High Pressure (Caution),Medium AC + High Pressure (Caution)
2,ChIJ-8DbTpYJxkcRF6GXegHzFe8,Hotel Hermitage Amsterdam,15.403297,23.963324,26.182952,Weesperbuurt,Weesperbuurt/Plantage,POINT (4.90361 52.36444),"POLYGON((4.901217 52.365779,4.901557 52.365162...","MULTIPOLYGON (((121885.88 486630.49, 121909.4 ...",...,112919.400811,Medium,Medium,Medium,Low,Medium,High,Medium AC + Low Pressure (Opportunity),Medium AC + Medium Pressure (Balanced),Medium AC + High Pressure (Caution)
3,ChIJ-RAyRD0KxkcRG4koK51opDE,Hotel Novotel Amsterdam City,24.843087,27.637239,27.644549,De Klenckebuurt,Buitenveldert-Oost,POINT (4.88852 52.33372),"POLYGON((4.878652 52.335088,4.879167 52.334426...","MULTIPOLYGON (((120717.778 483020.624, 120737....",...,16183.229403,High,High,High,Low,Low,Low,High AC + Low Pressure (Potential),High AC + Low Pressure (Potential),High AC + Low Pressure (Potential)
4,ChIJ-RS-x8cJxkcRix6HMEVbC0c,Avenue Hotel,8.965566,22.674889,25.467650,Nieuwendijk-Noord,Burgwallen-Nieuwe Zijde,POINT (4.89465 52.37665),"POLYGON((4.893582 52.376221,4.8951 52.376206,4...","MULTIPOLYGON (((121152.25 487850.511, 121172.1...",...,106913.368963,Low,Low,Low,High,High,Medium,Low AC + High Pressure (Critical),Low AC + High Pressure (Critical),Low AC + Medium Pressure (Monitor)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
412,ChIJzUkvQZUJxkcR7vLn27sChg0,SWEETS hotel Westerdoksbrug,13.396704,21.903330,23.892418,Westerdokseiland,Haarlemmerbuurt,POINT (4.89335 52.38333),"POLYGON((4.890186 52.382395,4.890637 52.382179...","MULTIPOLYGON (((121271.432 488352.305, 121281....",...,80992.273968,Low,Low,Low,Low,Medium,Medium,Low AC + Low Pressure (Limited Interest),Low AC + Medium Pressure (Monitor),Low AC + Medium Pressure (Monitor)
413,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,30.882961,30.704666,27.029511,Riekerpolder,Sloten/Nieuw-Sloten,POINT (4.8248 52.34177),"POLYGON((4.819922 52.33852,4.832922 52.338426,...","MULTIPOLYGON (((116430.193 483704.787, 116440....",...,93.172230,High,High,High,Low,Low,Low,High AC + Low Pressure (Potential),High AC + Low Pressure (Potential),High AC + Low Pressure (Potential)
414,ChIJzVtZ7PkJxkcRFn8elAS0PHg,The Delphi - Amsterdam Townhouse,21.154826,21.546693,29.000832,Minervabuurt-Noord,Apollobuurt,POINT (4.87693 52.35043),"POLYGON((4.866777 52.350394,4.867401 52.348677...","MULTIPOLYGON (((119910.206 484804.694, 119910....",...,59622.299074,High,Low,High,Medium,Medium,Low,High AC + Medium Pressure (Manageable),Low AC + Medium Pressure (Monitor),High AC + Low Pressure (Potential)
415,ChIJzcu9uuUJxkcRsinjE9Hf-L4,Sonder Park House,13.452959,27.180408,28.292420,P.C. Hooftbuurt,Museumkwartier,POINT (4.87785 52.36014),"POLYGON((4.877154 52.360101,4.878568 52.358153...","MULTIPOLYGON (((119969.752 486044.783, 119969....",...,90715.138903,Medium,High,High,Medium,Medium,Medium,Med

In [145]:
data = data.drop(columns = ["index_hotel_x", "pressure_x", "index_hotel_y", "pressure_y", "index_hotel", "pressure"])

In [146]:
data.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/hotels_ac_scores_now.csv")

In [256]:
#hotels_AC_buurt = hotels_AC.groupby(["Buurt", "Wijk", "WKT_LNG_LAT"]).mean().reset_index()
#hotels_AC_buurt.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/hotels_angular_choice_buurt.csv")
#hotels_AC.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/hotels_angular_choice_hotels.csv")

## Movement

In [263]:
from shapely.geometry import Point
movement = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/movement_from_hotel.csv")
movement = movement.rename(columns = {"place_id_hotel":"place_id", "place_name_hotel":"name", 
                                      "latitude_visited":"latitude", "longitude_visited":"longitude"})

# Convert lat/lon to geometry (Point objects)
movement['geometry'] = movement.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)

# Create a GeoDataFrame
movement = gpd.GeoDataFrame(movement, geometry='geometry', crs="EPSG:4326")
movement

,place_id,name,main_category_visited,latitude,longitude,geometry
0,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Locatie voor live muziek,52.362162,4.883806,POINT (4.88381 52.36216)
1,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Haute-cuisinerestaurant,52.369279,4.884023,POINT (4.88402 52.36928)
2,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Haute-cuisinerestaurant,52.369279,4.884023,POINT (4.88402 52.36928)
3,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Broodjeswinkel,52.370469,4.883942,POINT (4.88394 52.37047)
4,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,Snackbar,52.368827,4.884062,POINT (4.88406 52.36883)
...,...,...,...,...,...,...
881551,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,Kunstmuseum,52.358076,4.881205,POINT (4.88121 52.35808)
881552,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,Kunstmuseum,52.358011,4.879755,POINT (4.87976 52.35801)
881553,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,Brasserie,52.355639,4.870725,POINT (4.87073 52.35564)
881554,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,Bar,52.366458,4.900025,POINT (4.90002 52.36646)


In [265]:
hotels_tt_movement = hotels_tt[["place_id", "name", "walking_5min", "walking_15min", "walking_25min"]]
hotels_tt_movement = hotels_tt_movement.merge(movement, on = ["place_id","name"])

In [266]:
hotels_tt_movement["within_walking_5min"] = hotels_tt_movement.apply(lambda row: row.geometry.within(row.walking_5min) if row.walking_5min else False, axis=1)
hotels_tt_movement["within_walking_15min"] = hotels_tt_movement.apply(lambda row: row.geometry.within(row.walking_15min) if row.walking_15min else False, axis=1)
hotels_tt_movement["within_walking_25min"] = hotels_tt_movement.apply(lambda row: row.geometry.within(row.walking_25min) if row.walking_25min else False, axis=1)

In [278]:
hotel_reachability = hotels_tt_movement.groupby(["place_id", "name"]).agg(
    total_points=("geometry", "count"),
    within_5min=("within_walking_5min", "sum"),
    within_15min=("within_walking_15min", "sum"),
    within_25min=("within_walking_25min", "sum")
)

# Convert to percentage
hotel_reachability["percent_within_5min"] = (hotel_reachability["within_5min"] / hotel_reachability["total_points"]) * 100
hotel_reachability["percent_within_15min"] = (hotel_reachability["within_15min"] / hotel_reachability["total_points"]) * 100
hotel_reachability["percent_within_25min"] = (hotel_reachability["within_25min"] / hotel_reachability["total_points"]) * 100

hotel_reachability = hotel_reachability.reset_index()

hotel_reachability = hotels_AC.merge(hotel_reachability, on = ["place_id", "name"], how = "inner")
#hotel_reachability.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/hotels_all_data.csv")

## Cores

In [8]:
cores_tt = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/cores_traveltime.csv", index_col = 0)
space_syntax = gpd.read_file("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/new_space_syntax_pst.gpkg")

In [9]:
#25min 2000m
cores_tt["walking_25min"] = cores_tt["walking_25min"].apply(lambda x: loads(x))
cores_tt_25min = gpd.GeoDataFrame(cores_tt, geometry = "walking_25min", crs = "EPSG:4326")[["place", "walking_25min"]]
cores_tt_25min = cores_tt_25min.to_crs("EPSG:28992")
#15min 1200m
cores_tt["walking_15min"] = cores_tt["walking_15min"].apply(lambda x: loads(x))
cores_tt_15min = gpd.GeoDataFrame(cores_tt, geometry = "walking_15min", crs = "EPSG:4326")[["place", "walking_15min"]]
cores_tt_15min = cores_tt_15min.to_crs("EPSG:28992")
#5min 400m
cores_tt["walking_5min"] = cores_tt["walking_5min"].apply(lambda x: loads(x))
cores_tt_5min = gpd.GeoDataFrame(cores_tt, geometry = "walking_5min", crs = "EPSG:4326")[["place", "walking_5min"]]
cores_tt_5min = cores_tt_5min.to_crs("EPSG:28992")

In [10]:
space_syntax_h2000 = space_syntax[space_syntax['AC_w2k_NACH'] >= space_syntax["AC_w2k_NACH"].quantile(0.75)]
space_syntax_h1200 = space_syntax[space_syntax['AC_w1200_NACH'] >= space_syntax["AC_w1200_NACH"].quantile(0.75)]
space_syntax_h400 = space_syntax[space_syntax['AC_w400_NACH'] >= space_syntax["AC_w400_NACH"].quantile(0.75)]

In [11]:
# Getting the lines in every polygon and all high scoring lines on A.C.
cores_clip_5min = cores_tt_5min.overlay(space_syntax, how = "intersection", keep_geom_type = False)
cores_clip_5min_h = cores_tt_5min.overlay(space_syntax_h400, how = "intersection", keep_geom_type = False)

cores_clip_15min = cores_tt_15min.overlay(space_syntax, how = "intersection", keep_geom_type = False)
cores_clip_15min_h = cores_tt_15min.overlay(space_syntax_h1200, how = "intersection", keep_geom_type = False)

cores_clip_25min = cores_tt_25min.overlay(space_syntax, how = "intersection", keep_geom_type = False)
cores_clip_25min_h = cores_tt_25min.overlay(space_syntax_h2000, how = "intersection", keep_geom_type = False)

In [15]:
#calculting length of network and length of high A.C. network
cores_clip_5min_h["length line high"] = cores_clip_5min_h["geometry"].length
cores_clip_5min["length line"] = cores_clip_5min["geometry"].length
cores_5min_h_length = cores_clip_5min_h.groupby("place")["length line high"].sum().reset_index()
cores_5min_length = cores_clip_5min.groupby("place")["length line"].sum().reset_index()

cores_clip_15min_h["length line high"] = cores_clip_15min_h["geometry"].length
cores_clip_15min["length line"] = cores_clip_15min["geometry"].length
cores_15min_h_length = cores_clip_15min_h.groupby("place")["length line high"].sum().reset_index()
cores_15min_length = cores_clip_15min.groupby("place")["length line"].sum().reset_index()

cores_clip_25min_h["length line high"] = cores_clip_25min_h["geometry"].length
cores_clip_25min["length line"] = cores_clip_25min["geometry"].length
cores_25min_h_length = cores_clip_25min_h.groupby("place")["length line high"].sum().reset_index()
cores_25min_length = cores_clip_25min.groupby("place")["length line"].sum().reset_index()

In [16]:
cores_5min_length = cores_5min_h_length.merge(cores_5min_length, on = "place", how = "inner")
cores_15min_length = cores_15min_h_length.merge(cores_15min_length, on = "place", how = "inner")
cores_25min_length = cores_25min_h_length.merge(cores_25min_length, on = "place", how = "inner")

In [17]:
cores_5min_length["5min % high A.C."] = cores_5min_length["length line high"] / cores_5min_length["length line"] * 100
cores_15min_length["15min % high A.C."] = cores_15min_length["length line high"] / cores_15min_length["length line"] * 100
cores_25min_length["25min % high A.C."] = cores_25min_length["length line high"] / cores_25min_length["length line"] * 100

In [18]:
cores_5min_length = cores_5min_length.drop(columns = ["length line high", "length line"])
cores_15min_length = cores_15min_length.drop(columns = ["length line high", "length line"])
cores_25min_length = cores_25min_length.drop(columns = ["length line high", "length line"])

In [19]:
cores_AC = cores_5min_length.merge(cores_15min_length, on = "place", how = "inner")
cores_AC = cores_AC.merge(cores_25min_length, on = "place", how = "inner")
cores_AC

,place,5min % high A.C.,15min % high A.C.,25min % high A.C.
0,Bijlmer Arena,13.737573,21.810834,21.112626
1,Buikslotermeerplein,13.352358,20.894149,24.000897
2,Osdorpplein,15.852173,24.928870,25.763806
3,Sloterdijk Centrum,11.613181,25.247555,27.113710
4,Zuidas,17.021149,27.101810,28.763227


In [20]:
# Read data with info about locations and time
times = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/times.csv")
coordinates_places = times[["place_id", "coordinates"]]

# Parse the 'coordinates' column into separate latitude and longitude columns
times[['latitude', 'longitude']] = coordinates_places['coordinates'].apply(
    lambda x: pd.Series(json.loads(x))
)

# Drop the original 'coordinates' column for clarity
times = times.drop(columns='coordinates')

# Load the file with neighbourhood info
buurt = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/INDELING_BUURT (1).csv", delimiter=';')[["Buurt", "Wijk", "WKT_LNG_LAT"]]


# Load the points CSV
times_gdf = gpd.GeoDataFrame(
    times,
    geometry=gpd.points_from_xy(times['longitude'], times['latitude']),
    crs="EPSG:4326"  # Ensure the correct CRS, usually WGS84
)

# Load the polygons CSV
buurt['geometry'] = buurt['WKT_LNG_LAT'].apply(wkt.loads)
buurt_gdf = gpd.GeoDataFrame(buurt, geometry='geometry', crs="EPSG:4326")

# Perform spatial join
times_gdf = gpd.sjoin(times_gdf, buurt_gdf, how="left", predicate="within")
times_gdf = times_gdf.drop(columns = ["index_right", ])
times_gdf = times_gdf[["place_id", "name", "Buurt", "Wijk", "geometry", "WKT_LNG_LAT"]]

In [22]:
prs = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/pressure_tourism.csv", index_col=0)
plc_prs = prs.groupby(["place_id", "name"])["pressure"].sum().reset_index()
plc_prs = plc_prs.merge(times_gdf, on = ["place_id", "name"])
plc_prs = gpd.GeoDataFrame(plc_prs, geometry = "geometry", crs = "EPSG:4326")
plc_prs = plc_prs.to_crs("EPSG:28992")

In [23]:
#cores_clip_5min = cores_tt_5min.overlay(plc_prs, how = "intersection", keep_geom_type = False)
cores_pressure_5min = gpd.sjoin(plc_prs, cores_tt_5min, how="inner", predicate="within", lsuffix = "place", rsuffix = "core")
cores_pressure_15min = gpd.sjoin(plc_prs, cores_tt_15min, how="inner", predicate="within", lsuffix = "place", rsuffix = "core")
cores_pressure_25min = gpd.sjoin(plc_prs, cores_tt_25min, how="inner", predicate="within", lsuffix = "place", rsuffix = "core")


In [28]:
cores_pressure_5min = cores_pressure_5min.groupby("place")["pressure"].sum().reset_index()
cores_pressure_5min = cores_pressure_5min.rename(columns = {"pressure":"pressure 5min"})
cores_pressure_15min = cores_pressure_15min.groupby("place")["pressure"].sum().reset_index()
cores_pressure_15min = cores_pressure_15min.rename(columns = {"pressure":"pressure 15min"})
cores_pressure_25min = cores_pressure_25min.groupby("place")["pressure"].sum().reset_index()
cores_pressure_25min = cores_pressure_25min.rename(columns = {"pressure":"pressure 25min"})

In [31]:
cores_AC = cores_AC.merge(cores_pressure_5min, on = "place", how = "inner") 
cores_AC = cores_AC.merge(cores_pressure_15min, on = "place", how = "inner") 
cores_AC = cores_AC.merge(cores_pressure_25min, on = "place", how = "inner") 
cores_AC

,place,5min % high A.C.,15min % high A.C.,25min % high A.C.,pressure 5min,pressure 15min,pressure 25min
0,Bijlmer Arena,13.737573,21.810834,21.112626,15833.142027,32967.977197,33635.302746
1,Buikslotermeerplein,13.352358,20.894149,24.000897,7543.216576,7909.884175,9788.276654
2,Osdorpplein,15.852173,24.928870,25.763806,1495.623911,5383.960917,5829.059584
3,Sloterdijk Centrum,11.613181,25.247555,27.113710,1481.351203,2825.590747,4482.671874
4,Zuidas,17.021149,27.101810,28.763227,349.760072,5338.648029,24781.895302


In [33]:
#cores_AC_buurt = cores_AC.groupby(["Buurt", "Wijk", "WKT_LNG_LAT"]).mean().reset_index()
#cores_AC_buurt.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax....csv")
cores_AC.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/corec_AC.csv")

In [ ]:
test_times

### Wijken

In [6]:
wijken_area = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/INDELING_WIJK.csv", delimiter = ";")
space_syntax = gpd.read_file("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/new_space_syntax_pst.gpkg")

In [8]:
#25min 2000m
wijken_area["WKT_LNG_LAT"] = wijken_area["WKT_LNG_LAT"].apply(lambda x: loads(x))
wijken_area = gpd.GeoDataFrame(wijken_area, geometry = "WKT_LNG_LAT", 
                                  crs = "EPSG:4326")[["Wijk", "Oppervlakte_m2", "WKT_LNG_LAT"]]
wijken_area = wijken_area.to_crs("EPSG:28992")

In [11]:
space_syntax_h2000 = space_syntax[space_syntax['AC_w2k_NACH'] >= space_syntax["AC_w2k_NACH"].quantile(0.75)]
space_syntax_h1200 = space_syntax[space_syntax['AC_w1200_NACH'] >= space_syntax["AC_w1200_NACH"].quantile(0.75)]
space_syntax_h400 = space_syntax[space_syntax['AC_w400_NACH'] >= space_syntax["AC_w400_NACH"].quantile(0.75)]

In [12]:
# Getting the lines in every polygon and all high scoring lines on A.C.
wijken_ss_all = wijken_area.overlay(space_syntax, how = "intersection", keep_geom_type = False)
wijken_ss_h400 = wijken_area.overlay(space_syntax_h400, how = "intersection", keep_geom_type = False)
wijken_ss_h1200 = wijken_area.overlay(space_syntax_h1200, how = "intersection", keep_geom_type = False)
wijken_ss_h2000 = wijken_area.overlay(space_syntax_h2000, how = "intersection", keep_geom_type = False)

In [16]:
wijken_ss_h2000

,Wijk,Oppervlakte_m2,id,AC_w400_NACH,AC_w400_N,AC_w400_TD,AC_w400_MD,AI_w400_h,AI_w400_N,AI_w400_TD,...,AC_w2k_NACH,AC_w2k_N,AC_w2k_TD,AC_w2k_MD,AC_w1200_NACH,AC_w1200_N,AC_w1200_TD,AC_w1200_MD,length line,geometry
0,Da Costabuurt,257542,94837,1.316214,579,1982.739746,3.430346,209.483109,579,1599.324707,...,1.245678,11465,99417.179688,8.672120,1.271083,3586,23247.033203,6.484528,5.393766,"LINESTRING (120045.140 487231.350, 120040.100 ..."
1,Da Costabuurt,257542,94838,1.312821,588,2012.994995,3.429293,213.831955,588,1615.895874,...,1.248941,11258,96336.726562,8.557940,1.273060,3564,22838.681641,6.409958,7.072461,"LINESTRING (120040.100 487229.430, 120033.500 ..."
2,Da Costabuurt,257542,94839,1.311741,589,2018.758423,3.433263,214.933090,589,1613.088379,...,1.252633,11027,93026.117188,8.436978,1.276714,3522,22156.320312,6.292622,10.158780,"LINESTRING (120033.500 487226.890, 120024.500 ..."
3,Da Costabuurt,257542,94840,1.315262,593,1976.132812,3.338062,223.425339,594,1578.212158,...,1.253047,10973,92613.265625,8.440874,1.280980,3488,21413.902344,6.141067,34.999316,"LINESTRING (120024.500 487222.180, 119991.740 ..."
4,Da Costabuurt,257542,94841,1.320781,594,1931.973145,3.257965,227.571014,594,1549.443481,...,1.248926,11160,95234.093750,8.534286,1.284344,3464,20821.451172,6.012547,27.657187,"LINESTRING (119991.740 487209.870, 119965.790 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49844,Bellamybuurt,272774,321338,1.313234,325,1158.607544,3.575949,104.279366,324,1005.680481,...,1.324319,10135,76011.718750,7.500663,1.314289,3101,15252.215820,4.920070,39.213904,"LINESTRING (119153.480 486497.050, 119171.720 ..."
49845,Bellamybuurt,272774,321339,1.160241,274,931.250366,3.411174,91.297401,274,821.323486,...,1.320901,9989,75183.039062,7.527337,1.289412,3042,15220.150391,5.004982,72.191601,"LINESTRING (119171.720 486531.760, 119205.920 ..."
49846,Bellamybuurt,272774,321340,1.079106,210,609.601501,2.916754,81.102745,210,542.754700,...,1.309693,9940,74270.250000,7.472608,1.250484,2945,14681.783203,4.987019,94.686136,"LINESTRING (119205.920 486595.330, 119252.690 ..."
49847,Bellamybuurt,272774,321341,1.003528,191,539.832947,2.841226,76.997025,191,472.797546,...,1.313429,9673,70970.593750,7.337737,1.251039,2794,14028.160156,5.022614,60.449548,"LINESTRING (119252.690 486677.650, 119282.750 ..."


In [17]:
#calculting length of network and length of high A.C. network
wijken_ss_all["length line"] = wijken_ss_all["geometry"].length
wijken_ss_h400["length line high"] = wijken_ss_h400["geometry"].length
wijken_ss_h1200["length line high"] = wijken_ss_h1200["geometry"].length
wijken_ss_h2000["length line high"] = wijken_ss_h2000["geometry"].length


wijken_ss_all_length = wijken_ss_all.groupby("Wijk")["length line"].sum().reset_index()
wijken_ss_h400_length = wijken_ss_h400.groupby("Wijk")["length line high"].sum().reset_index()
wijken_ss_h1200_length = wijken_ss_h1200.groupby("Wijk")["length line high"].sum().reset_index()
wijken_ss_h2000_length = wijken_ss_h2000.groupby("Wijk")["length line high"].sum().reset_index()

In [27]:
wijken_ss_h1200_length

,Wijk,length line high,length line
0,Aetsveld/Oostelijke Vechtoever,16402.433624,36182.954429
1,Amstel III/Bullewijk,19266.507431,74467.951472
2,Amsterdamse Poort e.o.,9405.861653,42369.149177
3,Apollobuurt,5976.633053,26508.437355
4,Banne Buiksloot,12647.960423,55898.807750
...,...,...,...
105,Westlandgracht,7816.541337,29970.937311
106,Willemspark,6765.945414,22346.499712
107,Zeeburgereiland/Bovendiep,18907.607454,41053.065944
108,Zuid Pijp,2170.906762,10886.030470


In [21]:
wijken_ss_h400_length = wijken_ss_h400_length.merge(wijken_ss_all_length, on = "Wijk", how = "inner")
wijken_ss_h1200_length = wijken_ss_h1200_length.merge(wijken_ss_all_length, on = "Wijk", how = "inner")
wijken_ss_h2000_length = wijken_ss_h2000_length.merge(wijken_ss_all_length, on = "Wijk", how = "inner")

In [28]:
wijken_ss_h400_length["5min % high A.C."] = wijken_ss_h400_length["length line high"] / wijken_ss_h400_length["length line"] * 100
wijken_ss_h1200_length["15min % high A.C."] = wijken_ss_h1200_length["length line high"] / wijken_ss_h1200_length["length line"] * 100
wijken_ss_h2000_length["25min % high A.C."] = wijken_ss_h2000_length["length line high"] / wijken_ss_h2000_length["length line"] * 100

In [29]:
wijken_ss_h400_length = wijken_ss_h400_length.drop(columns = ["length line high", "length line"])
wijken_ss_h1200_length = wijken_ss_h1200_length.drop(columns = ["length line high", "length line"])
wijken_ss_h2000_length = wijken_ss_h2000_length.drop(columns = ["length line high", "length line"])

In [30]:
wijken_AC = wijken_ss_h400_length.merge(wijken_ss_h1200_length, on = "Wijk", how = "inner")
wijken_AC = wijken_AC.merge(wijken_ss_h2000_length, on = "Wijk", how = "inner")
wijken_AC

,Wijk,5min % high A.C.,15min % high A.C.,25min % high A.C.
0,Aetsveld/Oostelijke Vechtoever,46.210612,45.331936,49.024875
1,Amstel III/Bullewijk,22.872319,25.872214,27.219288
2,Amsterdamse Poort e.o.,17.804262,22.199789,22.102638
3,Apollobuurt,17.179171,22.546154,25.423298
4,Banne Buiksloot,12.165007,22.626530,20.242069
...,...,...,...,...
105,Westlandgracht,18.669630,26.080403,26.795955
106,Willemspark,16.371097,30.277428,34.446541
107,Zeeburgereiland/Bovendiep,35.760325,46.056505,48.868035
108,Zuid Pijp,15.848831,19.942134,21.881036


In [32]:
#wijken_AC.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/wijken_ac.csv")

## End

### Hotels

In [148]:
hotels_tt = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/hotels_traveltime.csv")
space_syntax = gpd.read_file("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/new_scenario_space_syntax_pst.gpkg")

In [152]:
#25min 2000m
hotels_tt["walking_25min"] = hotels_tt["walking_25min"].apply(lambda x: loads(x))
hotels_tt_25min = gpd.GeoDataFrame(hotels_tt, geometry = "walking_25min", crs = "EPSG:4326")[["place_id", "name", "walking_25min"]]
hotels_tt_25min = hotels_tt_25min.to_crs("EPSG:28992")
#15min 1200m
hotels_tt["walking_15min"] = hotels_tt["walking_15min"].apply(lambda x: loads(x))
hotels_tt_15min = gpd.GeoDataFrame(hotels_tt, geometry = "walking_15min", crs = "EPSG:4326")[["place_id", "name", "walking_15min"]]
hotels_tt_15min = hotels_tt_15min.to_crs("EPSG:28992")
#5min 400m
hotels_tt["walking_5min"] = hotels_tt["walking_5min"].apply(lambda x: loads(x))
hotels_tt_5min = gpd.GeoDataFrame(hotels_tt, geometry = "walking_5min", crs = "EPSG:4326")[["place_id", "name", "walking_5min"]]
hotels_tt_5min = hotels_tt_5min.to_crs("EPSG:28992")

In [153]:
hotels_tt_5min

,place_id,name,walking_5min
0,ChIJg8bf-sIJxkcRsvcYZBHfZ14,The Dylan Amsterdam,"MULTIPOLYGON (((120469.393 486772.663, 120469...."
1,ChIJIYdU8sMJxkcRrbxZuGZOlro,The Hoxton Amsterdam,"MULTIPOLYGON (((120622.466 487263.316, 120628...."
2,ChIJozDTD8MJxkcRO_HxQ1B4wF4,Grand Canal Boutique Hotel,"MULTIPOLYGON (((120447.398 486972.134, 120437...."
3,ChIJz-L17MMJxkcRZdrqv-u57P4,Hotel HEGRA by Stanley Collection,"MULTIPOLYGON (((120672.946 487244.162, 120682...."
4,ChIJo8-ANMIJxkcRi1HeWcbBcmI,Ambassade Hotel - Amsterdam,"MULTIPOLYGON (((120722.674 486970.6, 120722.60..."
...,...,...,...
414,ChIJPcEtxHEKxkcRx84qVvhT_uE,Cityden Zuidas,"MULTIPOLYGON (((119771.99 481298.257, 119771.8..."
415,ChIJVWZIVwsKxkcR4vauw0vYxuk,"Holiday Inn Express Amsterdam - South, an IHG ...","MULTIPOLYGON (((119270.415 481998.926, 119310...."
416,ChIJC_KvV9jhxUcRgDzDAbDHkBw,Amsterdam Forest Hotel,"MULTIPOLYGON (((118720.524 480896.844, 118720...."
417,ChIJN8RMMqgLxkcRknxrSHEle1E,Aparthotel Adagio Amsterdam City South,"MULTIPOLYGON (((119710.081 481343.172, 119740...."


In [154]:
space_syntax_h2000 = space_syntax[space_syntax['AC_w2k_NACH'] >= space_syntax["AC_w2k_NACH"].quantile(0.75)]
space_syntax_h1200 = space_syntax[space_syntax['AC_w1200_NACH'] >= space_syntax["AC_w1200_NACH"].quantile(0.75)]
space_syntax_h400 = space_syntax[space_syntax['AC_w400_NACH'] >= space_syntax["AC_w400_NACH"].quantile(0.75)]

In [155]:
# Getting the lines in every polygon and all high scoring lines on A.C.
hotels_clip_5min = hotels_tt_5min.overlay(space_syntax, how = "intersection", keep_geom_type = False)
hotels_clip_5min_h = hotels_tt_5min.overlay(space_syntax_h400, how = "intersection", keep_geom_type = False)

hotels_clip_15min = hotels_tt_15min.overlay(space_syntax, how = "intersection", keep_geom_type = False)
hotels_clip_15min_h = hotels_tt_15min.overlay(space_syntax_h1200, how = "intersection", keep_geom_type = False)

hotels_clip_25min = hotels_tt_25min.overlay(space_syntax, how = "intersection", keep_geom_type = False)
hotels_clip_25min_h = hotels_tt_25min.overlay(space_syntax_h2000, how = "intersection", keep_geom_type = False)

In [156]:
#calculting length of network and length of high A.C. network
hotels_clip_5min_h["length line high"] = hotels_clip_5min_h["geometry"].length
hotels_clip_5min["length line"] = hotels_clip_5min["geometry"].length
hotels_5min_h_length = hotels_clip_5min_h.groupby(["place_id", "name"])["length line high"].sum().reset_index()
hotels_5min_length = hotels_clip_5min.groupby(["place_id", "name"])["length line"].sum().reset_index()

hotels_clip_15min_h["length line high"] = hotels_clip_15min_h["geometry"].length
hotels_clip_15min["length line"] = hotels_clip_15min["geometry"].length
hotels_15min_h_length = hotels_clip_15min_h.groupby(["place_id", "name"])["length line high"].sum().reset_index()
hotels_15min_length = hotels_clip_15min.groupby(["place_id", "name"])["length line"].sum().reset_index()

hotels_clip_25min_h["length line high"] = hotels_clip_25min_h["geometry"].length
hotels_clip_25min["length line"] = hotels_clip_25min["geometry"].length
hotels_25min_h_length = hotels_clip_25min_h.groupby(["place_id", "name"])["length line high"].sum().reset_index()
hotels_25min_length = hotels_clip_25min.groupby(["place_id", "name"])["length line"].sum().reset_index()

In [157]:
hotels_5min_length = hotels_5min_h_length.merge(hotels_5min_length, on = ["place_id", "name"], how = "inner")
hotels_15min_length = hotels_15min_h_length.merge(hotels_15min_length, on = ["place_id", "name"], how = "inner")
hotels_25min_length = hotels_25min_h_length.merge(hotels_25min_length, on = ["place_id", "name"], how = "inner")

In [158]:
hotels_5min_length["5min % high A.C."] = hotels_5min_length["length line high"] / hotels_5min_length["length line"] * 100
hotels_15min_length["15min % high A.C."] = hotels_15min_length["length line high"] / hotels_15min_length["length line"] * 100
hotels_25min_length["25min % high A.C."] = hotels_25min_length["length line high"] / hotels_25min_length["length line"] * 100

In [159]:
hotels_5min_length = hotels_5min_length.drop(columns = ["length line high", "length line"])
hotels_15min_length = hotels_15min_length.drop(columns = ["length line high", "length line"])
hotels_25min_length = hotels_25min_length.drop(columns = ["length line high", "length line"])

In [160]:
hotels_AC = hotels_5min_length.merge(hotels_15min_length, on = ["place_id", "name"], how = "inner")
hotels_AC = hotels_AC.merge(hotels_25min_length, on = ["place_id", "name"], how = "inner")
hotels_AC

,place_id,name,5min % high A.C.,15min % high A.C.,25min % high A.C.
0,ChIJ-1hUL74JxkcRnLpTBTbfH5o,Eden Hotel Amsterdam,18.456850,23.202382,26.058997
1,ChIJ-8C0ZsQJxkcR6f0pc6CA3IM,Hotel De Westertoren,8.903487,23.484635,25.687926
2,ChIJ-8DbTpYJxkcRF6GXegHzFe8,Hotel Hermitage Amsterdam,15.403297,23.970916,26.196073
3,ChIJ-RAyRD0KxkcRG4koK51opDE,Hotel Novotel Amsterdam City,24.843087,27.637239,27.644549
4,ChIJ-RS-x8cJxkcRix6HMEVbC0c,Avenue Hotel,8.965566,22.674889,25.515208
...,...,...,...,...,...
412,ChIJzUkvQZUJxkcR7vLn27sChg0,SWEETS hotel Westerdoksbrug,13.396704,21.903330,23.933886
413,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,30.882961,30.761256,26.724162
414,ChIJzVtZ7PkJxkcRFn8elAS0PHg,The Delphi - Amsterdam Townhouse,21.154826,21.546958,28.977766
415,ChIJzcu9uuUJxkcRsinjE9Hf-L4,Sonder Park House,13.452959,27.180408,28.321815


In [161]:
# Read data with info about locations and time
times = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/times.csv")
coordinates_places = times[["place_id", "coordinates"]]

# Parse the 'coordinates' column into separate latitude and longitude columns
times[['latitude', 'longitude']] = coordinates_places['coordinates'].apply(
    lambda x: pd.Series(json.loads(x))
)

# Drop the original 'coordinates' column for clarity
times = times.drop(columns='coordinates')

# Load the file with neighbourhood info
buurt = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/INDELING_BUURT (1).csv", delimiter=';')[["Buurt", "Wijk", "WKT_LNG_LAT"]]


# Load the points CSV
times_gdf = gpd.GeoDataFrame(
    times,
    geometry=gpd.points_from_xy(times['longitude'], times['latitude']),
    crs="EPSG:4326"  # Ensure the correct CRS, usually WGS84
)

# Load the polygons CSV
buurt['geometry'] = buurt['WKT_LNG_LAT'].apply(wkt.loads)
buurt_gdf = gpd.GeoDataFrame(buurt, geometry='geometry', crs="EPSG:4326")

# Perform spatial join
times_gdf = gpd.sjoin(times_gdf, buurt_gdf, how="left", predicate="within")
times_gdf = times_gdf.drop(columns = ["index_right", ])
times_gdf = times_gdf[["place_id", "name", "Buurt", "Wijk", "geometry", "WKT_LNG_LAT"]]

In [162]:
hotels_AC = hotels_AC.merge(times_gdf, on = ["place_id", "name"], how = "inner")
hotels_AC

,place_id,name,5min % high A.C.,15min % high A.C.,25min % high A.C.,Buurt,Wijk,geometry,WKT_LNG_LAT
0,ChIJ-1hUL74JxkcRnLpTBTbfH5o,Eden Hotel Amsterdam,18.456850,23.202382,26.058997,Rembrandtplein e.o.,Grachtengordel-Zuid,POINT (4.89869 52.36684),"POLYGON((4.892818 52.36518,4.895874 52.364754,..."
1,ChIJ-8C0ZsQJxkcR6f0pc6CA3IM,Hotel De Westertoren,8.903487,23.484635,25.687926,Felix Meritisbuurt,Grachtengordel-West,POINT (4.88614 52.37346),"POLYGON((4.882614 52.368932,4.884411 52.368862..."
2,ChIJ-8DbTpYJxkcRF6GXegHzFe8,Hotel Hermitage Amsterdam,15.403297,23.970916,26.196073,Weesperbuurt,Weesperbuurt/Plantage,POINT (4.90361 52.36444),"POLYGON((4.901217 52.365779,4.901557 52.365162..."
3,ChIJ-RAyRD0KxkcRG4koK51opDE,Hotel Novotel Amsterdam City,24.843087,27.637239,27.644549,De Klenckebuurt,Buitenveldert-Oost,POINT (4.88852 52.33372),"POLYGON((4.878652 52.335088,4.879167 52.334426..."
4,ChIJ-RS-x8cJxkcRix6HMEVbC0c,Avenue Hotel,8.965566,22.674889,25.515208,Nieuwendijk-Noord,Burgwallen-Nieuwe Zijde,POINT (4.89465 52.37665),"POLYGON((4.893582 52.376221,4.8951 52.376206,4..."
...,...,...,...,...,...,...,...,...,...
412,ChIJzUkvQZUJxkcR7vLn27sChg0,SWEETS hotel Westerdoksbrug,13.396704,21.903330,23.933886,Westerdokseiland,Haarlemmerbuurt,POINT (4.89335 52.38333),"POLYGON((4.890186 52.382395,4.890637 52.382179..."
413,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,30.882961,30.761256,26.724162,Riekerpolder,Sloten/Nieuw-Sloten,POINT (4.8248 52.34177),"POLYGON((4.819922 52.33852,4.832922 52.338426,..."
414,ChIJzVtZ7PkJxkcRFn8elAS0PHg,The Delphi - Amsterdam Townhouse,21.154826,21.546958,28.977766,Minervabuurt-Noord,Apollobuurt,POINT (4.87693 52.35043),"POLYGON((4.866777 52.350394,4.867401 52.348677..."
415,ChIJzcu9uuUJxkcRsinjE9Hf-L4,Sonder Park House,13.452959,27.180408,28.321815,P.C. Hooftbuurt,Museumkwartier,POINT (4.87785 52.36014),"POLYGON((4.877154 52.360101,4.878568 52.358153..."


In [158]:
#space syntax high values
#space_syntax_h400[["id", "AC_w400_NACH", "geometry"]].to_file("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/space_syntax_h400.gpkg", driver = "GPKG")
#space_syntax_h1200[["id", "AC_w1200_NACH", "geometry"]].to_file("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/space_syntax_h1200.gpkg", driver = "GPKG")
#space_syntax_h2000[["id", "AC_w2k_NACH", "geometry"]].to_file("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/space_syntax_h2000.gpkg", driver = "GPKG")

In [163]:
prs = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/pressure_tourism.csv", index_col=0)
plc_prs = prs.groupby(["place_id", "name"])["pressure"].sum().reset_index()
plc_prs = plc_prs.merge(times_gdf, on = ["place_id", "name"])
plc_prs = gpd.GeoDataFrame(plc_prs, geometry = "geometry", crs = "EPSG:4326")
plc_prs = plc_prs.to_crs("EPSG:28992")

In [164]:
#hotels_clip_5min = hotels_tt_5min.overlay(plc_prs, how = "intersection", keep_geom_type = False)
hotels_pressure_5min = gpd.sjoin(plc_prs, hotels_tt_5min, how="inner", predicate="within", lsuffix = "place", rsuffix = "hotel")
hotels_pressure_15min = gpd.sjoin(plc_prs, hotels_tt_15min, how="inner", predicate="within", lsuffix = "place", rsuffix = "hotel")
hotels_pressure_25min = gpd.sjoin(plc_prs, hotels_tt_25min, how="inner", predicate="within", lsuffix = "place", rsuffix = "hotel")


In [165]:
# Add hotel_id if not present
hotels_tt_5min["index_hotel"] = hotels_tt_5min.index
# Group by hotel polygon and sum the pressure
pressure_sums_5min = hotels_pressure_5min.groupby("index_hotel")["pressure"].sum().reset_index()
# Add hotels with no pressure
hotels_pressure_5min_h = hotels_tt_5min.merge(pressure_sums_5min, on="index_hotel", how="left")
hotels_pressure_5min_h["pressure 5min"] = hotels_pressure_5min_h["pressure"].fillna(0)


# Add hotel_id if not present
hotels_tt_15min["index_hotel"] = hotels_tt_15min.index
# Group by hotel polygon and sum the pressure
pressure_sums_15min = hotels_pressure_15min.groupby("index_hotel")["pressure"].sum().reset_index()
# Add hotels with no pressure
hotels_pressure_15min_h = hotels_tt_15min.merge(pressure_sums_15min, on="index_hotel", how="left")
hotels_pressure_15min_h["pressure 15min"] = hotels_pressure_15min_h["pressure"].fillna(0)

# Add hotel_id if not present
hotels_tt_25min["index_hotel"] = hotels_tt_25min.index
# Group by hotel polygon and sum the pressure
pressure_sums_25min = hotels_pressure_25min.groupby("index_hotel")["pressure"].sum().reset_index()
# Add hotels with no pressure
hotels_pressure_25min_h = hotels_tt_25min.merge(pressure_sums_25min, on="index_hotel", how="left")
hotels_pressure_25min_h["pressure 25min"] = hotels_pressure_25min_h["pressure"].fillna(0)



In [166]:
hotels_AC = hotels_AC.merge(hotels_pressure_5min_h, on = ["place_id", "name"], how = "inner") 
hotels_AC = hotels_AC.merge(hotels_pressure_15min_h, on = ["place_id", "name"], how = "inner") 
hotels_AC = hotels_AC.merge(hotels_pressure_25min_h, on = ["place_id", "name"], how = "inner")

In [167]:
data = hotels_AC
# Define a function to categorize clearly based on percentiles
def categorize(series):
    low = series.quantile(0.33)
    high = series.quantile(0.66)
    
    def get_category(value):
        if value <= low:
            return 'Low'
        elif value <= high:
            return 'Medium'
        else:
            return 'High'
    
    return series.apply(get_category)

# Categorize % high Angular Choice clearly
for col in ['5min % high A.C.', '15min % high A.C.', '25min % high A.C.']:
    data[f'{col} Category'] = categorize(data[col])

# Categorize Tourist Pressure clearly
for col in ['pressure 5min', 'pressure 15min', 'pressure 25min']:
    data[f'{col} Category'] = categorize(data[col])

# Define function to create combined categories clearly
def combined_category(ac, pressure):
    if ac == 'High' and pressure == 'Low':
        return 'High AC + Low Pressure (Potential)'
    elif ac == 'High' and pressure == 'High':
        return 'High AC + High Pressure (Stress)'
    elif ac == 'Low' and pressure == 'High':
        return 'Low AC + High Pressure (Critical)'
    elif ac == 'Medium' and pressure == 'Medium':
        return 'Medium AC + Medium Pressure (Balanced)'
    elif ac == 'Low' and pressure == 'Low':
        return 'Low AC + Low Pressure (Limited Interest)'
    elif ac == 'High' and pressure == 'Medium':
        return 'High AC + Medium Pressure (Manageable)'
    elif ac == 'Medium' and pressure == 'High':
        return 'Medium AC + High Pressure (Caution)'
    elif ac == 'Medium' and pressure == 'Low':
        return 'Medium AC + Low Pressure (Opportunity)'
    elif ac == 'Low' and pressure == 'Medium':
        return 'Low AC + Medium Pressure (Monitor)'
    else:
        return 'Other'

# Apply this combined category clearly for each timescale
for time in ['5min', '15min', '25min']:
    ac_col = f'{time} % high A.C. Category'
    pressure_col = f'pressure {time} Category'
    data[f'Combined Category {time}'] = data.apply(lambda row: combined_category(row[ac_col], row[pressure_col]), axis=1)
    

# Display the first rows to check categorization
data


,place_id,name,5min % high A.C.,15min % high A.C.,25min % high A.C.,Buurt,Wijk,geometry,WKT_LNG_LAT,walking_5min,...,pressure 25min,5min % high A.C. Category,15min % high A.C. Category,25min % high A.C. Category,pressure 5min Category,pressure 15min Category,pressure 25min Category,Combined Category 5min,Combined Category 15min,Combined Category 25min
0,ChIJ-1hUL74JxkcRnLpTBTbfH5o,Eden Hotel Amsterdam,18.456850,23.202382,26.058997,Rembrandtplein e.o.,Grachtengordel-Zuid,POINT (4.89869 52.36684),"POLYGON((4.892818 52.36518,4.895874 52.364754,...","MULTIPOLYGON (((121474.683 486597.511, 121484....",...,126486.580990,Medium,Medium,Medium,Medium,Medium,High,Medium AC + Medium Pressure (Balanced),Medium AC + Medium Pressure (Balanced),Medium AC + High Pressure (Caution)
1,ChIJ-8C0ZsQJxkcR6f0pc6CA3IM,Hotel De Westertoren,8.903487,23.484635,25.687926,Felix Meritisbuurt,Grachtengordel-West,POINT (4.88614 52.37346),"POLYGON((4.882614 52.368932,4.884411 52.368862...","MULTIPOLYGON (((120673.337 487410.513, 120678....",...,118663.332018,Low,Medium,Medium,Medium,High,High,Low AC + Medium Pressure (Monitor),Medium AC + High Pressure (Caution),Medium AC + High Pressure (Caution)
2,ChIJ-8DbTpYJxkcRF6GXegHzFe8,Hotel Hermitage Amsterdam,15.403297,23.970916,26.196073,Weesperbuurt,Weesperbuurt/Plantage,POINT (4.90361 52.36444),"POLYGON((4.901217 52.365779,4.901557 52.365162...","MULTIPOLYGON (((121885.88 486630.49, 121909.4 ...",...,112919.400811,Medium,Medium,Medium,Low,Medium,High,Medium AC + Low Pressure (Opportunity),Medium AC + Medium Pressure (Balanced),Medium AC + High Pressure (Caution)
3,ChIJ-RAyRD0KxkcRG4koK51opDE,Hotel Novotel Amsterdam City,24.843087,27.637239,27.644549,De Klenckebuurt,Buitenveldert-Oost,POINT (4.88852 52.33372),"POLYGON((4.878652 52.335088,4.879167 52.334426...","MULTIPOLYGON (((120717.778 483020.624, 120737....",...,16183.229403,High,High,High,Low,Low,Low,High AC + Low Pressure (Potential),High AC + Low Pressure (Potential),High AC + Low Pressure (Potential)
4,ChIJ-RS-x8cJxkcRix6HMEVbC0c,Avenue Hotel,8.965566,22.674889,25.515208,Nieuwendijk-Noord,Burgwallen-Nieuwe Zijde,POINT (4.89465 52.37665),"POLYGON((4.893582 52.376221,4.8951 52.376206,4...","MULTIPOLYGON (((121152.25 487850.511, 121172.1...",...,106913.368963,Low,Low,Low,High,High,Medium,Low AC + High Pressure (Critical),Low AC + High Pressure (Critical),Low AC + Medium Pressure (Monitor)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
412,ChIJzUkvQZUJxkcR7vLn27sChg0,SWEETS hotel Westerdoksbrug,13.396704,21.903330,23.933886,Westerdokseiland,Haarlemmerbuurt,POINT (4.89335 52.38333),"POLYGON((4.890186 52.382395,4.890637 52.382179...","MULTIPOLYGON (((121271.432 488352.305, 121281....",...,80992.273968,Low,Low,Low,Low,Medium,Medium,Low AC + Low Pressure (Limited Interest),Low AC + Medium Pressure (Monitor),Low AC + Medium Pressure (Monitor)
413,ChIJzVN3QY_hxUcREIzfWnT4u94,Hotel Artemis Amsterdam,30.882961,30.761256,26.724162,Riekerpolder,Sloten/Nieuw-Sloten,POINT (4.8248 52.34177),"POLYGON((4.819922 52.33852,4.832922 52.338426,...","MULTIPOLYGON (((116430.193 483704.787, 116440....",...,93.172230,High,High,Medium,Low,Low,Low,High AC + Low Pressure (Potential),High AC + Low Pressure (Potential),Medium AC + Low Pressure (Opportunity)
414,ChIJzVtZ7PkJxkcRFn8elAS0PHg,The Delphi - Amsterdam Townhouse,21.154826,21.546958,28.977766,Minervabuurt-Noord,Apollobuurt,POINT (4.87693 52.35043),"POLYGON((4.866777 52.350394,4.867401 52.348677...","MULTIPOLYGON (((119910.206 484804.694, 119910....",...,59622.299074,High,Low,High,Medium,Medium,Low,High AC + Medium Pressure (Manageable),Low AC + Medium Pressure (Monitor),High AC + Low Pressure (Potential)
415,ChIJzcu9uuUJxkcRsinjE9Hf-L4,Sonder Park House,13.452959,27.180408,28.321815,P.C. Hooftbuurt,Museumkwartier,POINT (4.87785 52.36014),"POLYGON((4.877154 52.360101,4.878568 52.358153...","MULTIPOLYGON (((119969.752 486044.783, 119969....",...,90715.138903,Medium,High,High,Medium,Medium,Medi

In [168]:
data = data.drop(columns = ["index_hotel_x", "pressure_x", "index_hotel_y", "pressure_y", "index_hotel", "pressure"])

In [169]:
data.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/hotels_ac_scores_future.csv")

In [256]:
#hotels_AC_buurt = hotels_AC.groupby(["Buurt", "Wijk", "WKT_LNG_LAT"]).mean().reset_index()
#hotels_AC_buurt.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/hotels_angular_choice_buurt.csv")
#hotels_AC.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/space syntax/hotels_angular_choice_hotels.csv")